# Preprocess GraphCast Data

In [1]:
import re
import os
import glob
import shutil
import zipfile

import numpy as np
import pandas as pd
import xarray as xr

from pathlib import Path
from utils import DATA_ROOT, notify
from tqdm.contrib.concurrent import process_map
from dask.distributed import Client, LocalCluster

In [2]:
# Path to the folder containing the zip files
source_folder = os.path.join(DATA_ROOT, "input/GC")

# Path to the target folder
dest_folder = os.path.join(DATA_ROOT, "derived/GC")

ncpus = 120

## Functions

In [3]:
# Unzip file
def unzip(in_file, out_file):
    with zipfile.ZipFile(in_file, 'r') as zip_ref:
        zip_ref.extractall(out_file)

# Clip domain to Western US
def clip_domain(
    filepath, xmin=-125, xmax=-116.20, ymin=32, ymax=41.50
):
    ds = xr.open_dataset(filepath, decode_timedelta=True)

    new_file = filepath + '_clip'

    ds.isel(
        longitude=(xmin <= ds.longitude) & (ds.longitude <= xmax),
        latitude=(ymin <= ds.latitude) & (ds.latitude <= ymax),
    ).to_netcdf(new_file)

    return new_file

# Preprocess daily NC files into a zarr file
def preprocess_nc_files(ds):
    # Get the file path from ds encoding
    file_path = ds.encoding['source']

    # Determine the variable name based on file name
    var = file_path.split()[-1].split('/')[-1].split('_')[0]

    if var == '10u':
        var = 'u10'
    elif var == '10v':
        var = 'v10'
    elif var == '2t':
        var = 't2m'

    # Subset the dataset to only one variable
    ds = ds[[var]]

    # Remove the unnecessary dimension
    if var == 'tp':
        ds = ds.isel(time=0)
        ds = ds.drop_vars('time')

    # Determine initialization time
    match = re.search(r'graphcast_(\d{8})_(\d{4})', file_path)
    assert match, f'No match for {file_path}'
    date_str = match.group(1)
    hour_str = match.group(2)

    # Convert to pandas Timestamp
    init_time = pd.to_datetime(
        f'{date_str} {hour_str[:2]}:{hour_str[2:]}',
        format='%Y%m%d %H:%M'
    )

    # Get full variable name with vertical information
    full_variable_name = file_path.split('/')[-1].rstrip('.nc')

    # Create a new variable with the initialization and variable
    # dimensions to replace the old variable
    #
    ds['data'] = ds[var].expand_dims({
        'init': [init_time],
        'var': [full_variable_name]
    })

    ds = ds.drop_vars(var)
    ds['var'] = ds['var'].astype(str)

    # Fix dimension order
    ds = ds.transpose('init', 'step', 'var', 'latitude', 'longitude')

    return ds

def merge_as_zarr(dest_path):
    # Get valid NC files to merge
    all_files = glob.glob(os.path.join(dest_path, '*.nc'))
    
    all_files = [
        f for f in sorted(all_files)
        if 'lsm_surface_lvl0' not in f and 'z_surface_lvl0' not in f and 'msl_meanSea_lvl0' not in f
    ]
    
    ds = xr.open_mfdataset(
        all_files,
        preprocess=preprocess_nc_files,
        parallel=False,
        decode_cf=False,
    )
    
    ds = ds.chunk({
        'init': 1,
        'step': 1,
        'var': -1,
        'latitude': -1,
        'longitude': -1,
    })
    
    ds.to_zarr(
        dest_path + '.zarr',
        zarr_format=2,
    )

# Preprocess one file
def preprocess(
    file_name,
    source_folder=source_folder,
    dest_folder=dest_folder
):
    # Call the unzip routine
    zip_path = os.path.join(source_folder, file_name)
    dest_path = os.path.join(dest_folder, os.path.splitext(file_name)[0])
    unzip(zip_path, dest_path)

    # Call the clipping routine
    nc_files = [
        os.path.join(dest_path, f)
        for f in os.listdir(dest_path)
        if f.lower().endswith('.nc')
    ]

    for nc_file in nc_files:
        clipped_file = clip_domain(nc_file)
        os.remove(nc_file)
        os.rename(clipped_file, nc_file)

    # Call the merge routine
    merge_as_zarr(dest_path)
    shutil.rmtree(dest_path)

## File Preprocessing Workflow

In [4]:
# Make sure the unzipped folder exists
os.makedirs(dest_folder, exist_ok=True)

# Get all .zip files first
zip_files = [
    f for f in os.listdir(source_folder)
    if f.lower().endswith('.zip')
]

zip_files = sorted(zip_files)

In [5]:
%%time

# Wall time: 21min 27s
_ = process_map(preprocess, zip_files, max_workers=ncpus, chunksize=1)

  0%|          | 0/1827 [00:00<?, ?it/s]

CPU times: user 1.14 s, sys: 507 ms, total: 1.65 s
Wall time: 21min 27s


## Merge Zarr Files

In [6]:
zarr_files = glob.glob(os.path.join(dest_folder, '*_gpu.zarr'))
zarr_files = sorted(zarr_files)
print(f'There are {len(zarr_files)} zarr files to merge!')

There are 1827 zarr files to merge!


In [7]:
cluster = LocalCluster(
    n_workers=120,
    threads_per_worker=1,
    memory_limit='2G',
    dashboard_address=':41228',
)

client = Client(cluster)

In [8]:
with xr.open_mfdataset(zarr_files, parallel=True) as ds:
    _ = ds.to_zarr(
        os.path.join(DATA_ROOT, 'derived/GC/merged.zarr'),
        mode='w',
        zarr_format=2,
    )

In [9]:
_ = process_map(shutil.rmtree, zarr_files, max_workers=ncpus, chunksize=1)

  0%|          | 0/1827 [00:00<?, ?it/s]

## Resample to Daily Resolution

In [3]:
with xr.open_zarr(
    Path(DATA_ROOT) / 'derived/GC/merged.zarr',
) as ds_gc:
    
    # Split
    precip_ds = ds_gc.sel(var='tp_surface_lvl0')
    other_ds = ds_gc.sel(var=ds_gc['var'] != 'tp_surface_lvl0')
    
    # Define indices
    precip_day_indices = [
        (4, 0), (8, 4), (12, 8), (16, 12),
        (20, 16), (24, 20), (28, 24)
    ]
    
    other_day_slices = [
        slice(1, 5), slice(5, 9), slice(9, 13), slice(13, 17),
        slice(17, 21), slice(21, 25), slice(25, 29)]

    # Process precipitation
    precip_daily_list = [
        precip_ds['data'].isel(step=i) - (
            precip_ds['data'].isel(step=j).fillna(0)
            if j == 0
            else precip_ds['data'].isel(step=j)
            ) 
        for i, j in precip_day_indices
    ]
    
    precip_daily = xr.concat(
        [
            d.expand_dims(step=[day])
            for day, d in enumerate(precip_daily_list, 1)
        ],
        dim='step',
    ).expand_dims(var=['tp_surface_lvl0'])
    
    # Process other variables
    other_daily_list = [
        other_ds['data'].isel(step=s).mean(dim='step')
        for s in other_day_slices
    ]
    other_daily = xr.concat(
        [
            d.expand_dims(step=[day])
            for day, d in enumerate(other_daily_list, 1)
        ],
        dim='step',
    )
    other_daily = other_daily.assign_coords(var=other_ds['var'])
    
    # Combine
    combined_daily = xr.concat([precip_daily, other_daily], dim='var')
    
    # Proper step variable for daily resolution
    n_days = combined_daily.sizes['step']
    new_step = np.array([
        np.timedelta64(day, 'D')
        for day in range(1, n_days + 1)
    ])
    combined_daily = combined_daily.assign_coords(step=new_step)
    
    # New Dataset
    ds_gc_daily = xr.Dataset(
        {'data': combined_daily},
        coords={
            'init': ds_gc['init'],
            'step': combined_daily['step'],
            'var': combined_daily['var'],
            'latitude': ds_gc['latitude'],
            'longitude': ds_gc['longitude']
        }
    )
    
    ds_gc_daily = ds_gc_daily.chunk({
        'init': 1,
        'step': 1,
        'var': -1,
        'latitude': -1,
        'longitude': -1,
    })
    
    ds_gc_daily.to_zarr(
        os.path.join(DATA_ROOT, 'derived/GC/daily.zarr'),
        mode='w',
        align_chunks=True,
        zarr_format=2,
    )

In [4]:
notify('GC restructure is complete!')